# Alice Vision Meshroom pipeline

You need to download and install Meshroom binaries: https://alicevision.org/#meshroom

In [1]:
import subprocess
import os
os.environ["OPENCV_IO_ENABLE_OPENEXR"]="1"
import cv2
import shutil
from pathlib import Path
import numpy as np
import open3d as o3d
import gc

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Running Pipeline

In [2]:
meshroom_batch = r"C:\Users\gnoceras\Downloads\Softwares\Meshroom\Meshroom-2025.1.0\meshroom_batch.exe"

input_images = "../data"
output_dir = "../results/meshroom"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

def run_meshroom_command(cmd):
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    print(result.stdout)
    print(result.stderr)
    return result

cmd = [
    meshroom_batch,
    "--input", input_images,
    "--output", output_dir
]

run_meshroom_command(cmd)

Running: C:\Users\gnoceras\Downloads\Softwares\Meshroom\Meshroom-2025.1.0\meshroom_batch.exe --input ../data --output ../results/meshroom
[2026-05-11 13:58:43.347617] [0x000063ac] [trace]   Embedded OCIO configuration file: 'C:\Users\gnoceras\Downloads\Softwares\Meshroom\Meshroom-2025.1.0\aliceVision/share/aliceVision/config.ocio' found.
Program called with the following parameters:
 * allowSingleView = 1
 * colorProfileDatabase = "" (default)
 * defaultCameraModel = "" (default)
 * defaultDistortionModel = "" (default)
 * defaultFieldOfView = 45
 * defaultFocalLength = -1 (default)
 * defaultFocalRatio = 1 (default)
 * defaultOffsetX = 0 (default)
 * defaultOffsetY = 0 (default)
 * errorOnMissingColorProfile = 1 (default)
 * groupCameraFallback =  Unknown Type "enum EGroupCameraFallback"
 * imageFolder = "" (default)
 * input = "C:\Users\gnoceras\AppData\Local\Temp\tmptmfam579/CameraInit/4c471e04d91d618d3dddf7990cb9b81c599a7700\viewpoints.sfm"
 * lensCorrectionProfileInfo = "${ALICEVI

CompletedProcess(args=['C:\\Users\\gnoceras\\Downloads\\Softwares\\Meshroom\\Meshroom-2025.1.0\\meshroom_batch.exe', '--input', '../data', '--output', '../results/meshroom'], returncode=0, stdout='[2026-05-11 13:58:43.347617] [0x000063ac] [trace]   Embedded OCIO configuration file: \'C:\\Users\\gnoceras\\Downloads\\Softwares\\Meshroom\\Meshroom-2025.1.0\\aliceVision/share/aliceVision/config.ocio\' found.\nProgram called with the following parameters:\n * allowSingleView = 1\n * colorProfileDatabase = "" (default)\n * defaultCameraModel = "" (default)\n * defaultDistortionModel = "" (default)\n * defaultFieldOfView = 45\n * defaultFocalLength = -1 (default)\n * defaultFocalRatio = 1 (default)\n * defaultOffsetX = 0 (default)\n * defaultOffsetY = 0 (default)\n * errorOnMissingColorProfile = 1 (default)\n * groupCameraFallback =  Unknown Type "enum EGroupCameraFallback"\n * imageFolder = "" (default)\n * input = "C:\\Users\\gnoceras\\AppData\\Local\\Temp\\tmptmfam579/CameraInit/4c471e04d9

## Visualizing Results

In [ ]:
# Path to your Meshroom OBJ file (update as needed)
obj_path = Path("../results/meshroom_rec/texturedMesh.obj")
out_ply = Path("../results/meshroom_rec/reconstruction.ply")

In [ ]:
try:
    import imageio.v3 as iio
    print('Using imageio for EXR conversion')
except ImportError:
    raise ImportError('Please install imageio: pip install imageio[all]')

# Validate core files exist
if not obj_path.exists():
    raise FileNotFoundError(f'OBJ file not found: {obj_path}')

mtl_path = obj_path.with_suffix('.mtl')
if not mtl_path.exists():
    raise FileNotFoundError(f'MTL file not found: {mtl_path}')

def _convert_exr_to_png(exr_path: Path) -> Path:
    """Load EXR texture, tone-map to 8-bit, and write PNG."""
    print(f'Converting {exr_path.name} to PNG...')
    texture = iio.imread(exr_path)
    
    # Handle different data types and convert to 8-bit
    if texture.dtype in (np.float32, np.float64):
        # Use percentile-based normalization instead of hard clip
        p_low, p_high = np.percentile(texture[texture > 0], [1, 99]) if texture.max() > 0 else (0, 1)
        texture = np.clip((texture - p_low) / (p_high - p_low + 1e-8), 0.0, 1.0)
        texture = (texture * 255.0).astype(np.uint8)
    elif texture.dtype != np.uint8:
        texture = texture.astype(np.uint8)
    
    # Ensure RGB format
    if texture.ndim == 2:
        texture = np.stack([texture]*3, axis=-1)
    elif texture.shape[-1] == 4:
        texture = texture[..., :3]  # Remove alpha channel
    
    png_path = exr_path.with_suffix('.png')
    iio.imwrite(png_path, texture)
    print(f'  → Saved {png_path.name}')
    return png_path

# Update MTL so Open3D can locate PNG textures
with open(mtl_path, 'r', encoding='utf-8') as handle:
    mtl_lines = handle.readlines()

updated = False
for idx, line in enumerate(mtl_lines):
    tokens = line.strip().split(maxsplit=1)
    if tokens and tokens[0].lower() == 'map_kd' and len(tokens) == 2:
        tex_rel_path = tokens[1]
        tex_path = (mtl_path.parent / tex_rel_path).resolve()
        if tex_path.suffix.lower() == '.exr' and tex_path.exists():
            png_path = _convert_exr_to_png(Path(tex_path))
            rel_png = os.path.relpath(png_path, mtl_path.parent).replace('\\', '/')
            mtl_lines[idx] = f'map_Kd {rel_png}\n'
            updated = True

if updated:
    backup_path = mtl_path.with_suffix('.mtl.bak')
    if not backup_path.exists():
        shutil.copy2(mtl_path, backup_path)
    with open(mtl_path, 'w', encoding='utf-8') as handle:
        handle.writelines(mtl_lines)
    print(f'\nConverted EXR textures to PNG and updated MTL.')
    print(f'Original MTL backed up to: {backup_path}')
else:
    print('No EXR textures found to convert.')

In [5]:
# Load the mesh (Open3D will try to load textures if referenced in the .mtl file)
mesh = o3d.io.read_triangle_mesh(obj_path, enable_post_processing=True)

# Check if mesh has vertex colors or textures
if mesh.has_vertex_colors():
    print('Mesh has vertex colors.')
elif mesh.has_textures():
    print('Mesh has textures.')
else:
    print('Mesh has no vertex colors or textures.')

# Visualize the mesh
o3d.visualization.draw_geometries([mesh], window_name='Meshroom OBJ Visualization')

Mesh has textures.


In [7]:
if not obj_path.exists():
    raise FileNotFoundError(f'OBJ file not found: {obj_path}')

# load mesh (Open3D will read .mtl but may not load EXR textures)
mesh = o3d.io.read_triangle_mesh(str(obj_path), enable_post_processing=True)

# try to find texture referenced in .mtl
mtl_path = obj_path.with_suffix('.mtl')
texture_files = []
if mtl_path.exists():
    for line in mtl_path.read_text(encoding='utf-8', errors='ignore').splitlines():
        tokens = line.strip().split(maxsplit=1)
        if tokens and tokens[0].lower() == 'map_kd' and len(tokens) == 2:
            tex = (mtl_path.parent / tokens[1].strip()).resolve()
            if tex.exists():
                texture_files.append(tex)
            else:
                # try same name with .png if EXR was replaced earlier
                alt = tex.with_suffix('.png')
                if alt.exists():
                    texture_files.append(alt)

if len(texture_files) == 0:
    print("No texture file found in MTL. Attempting to use mesh textures (if any).")

# prefer first texture
tex_path = Path(texture_files[0]) if texture_files else None
if tex_path:
    print(f"Using texture: {tex_path}")
else:
    print("No usable texture found. Will attempt point sampling fallback.")

# helper: bilinear sample from image at uv (u,v in [0,1])
def sample_texture_bilinear(img, uv):
    h, w = img.shape[:2]
    u = np.clip(uv[:, 0], 0.0, 1.0) * (w - 1)
    v = np.clip(uv[:, 1], 0.0, 1.0) * (h - 1)
    # v origin correction (OBJ UV convention -> v=0 bottom); most images top-down so invert v
    v = (h - 1) - v
    x0 = np.floor(u).astype(np.int32)
    x1 = np.clip(x0 + 1, 0, w - 1)
    y0 = np.floor(v).astype(np.int32)
    y1 = np.clip(y0 + 1, 0, h - 1)
    wx = u - x0
    wy = v - y0
    c00 = img[y0, x0]
    c10 = img[y0, x1]
    c01 = img[y1, x0]
    c11 = img[y1, x1]
    c0 = c00 * (1 - wx)[:, None] + c10 * wx[:, None]
    c1 = c01 * (1 - wx)[:, None] + c11 * wx[:, None]
    c = c0 * (1 - wy)[:, None] + c1 * wy[:, None]
    return c.astype(np.float32) / 255.0

def bake_vertex_colors_from_texture(mesh, tex_img_path, batch_faces=50000):
    # ensure UVs exist
    if not hasattr(mesh, "triangle_uvs") or len(mesh.triangle_uvs) == 0:
        raise RuntimeError("Mesh has no triangle UVs (triangle_uvs). Cannot bake.")
    # load texture
    img = iio.imread(str(tex_img_path))
    if img.dtype != np.uint8:
        img = np.clip(img, 0.0, 1.0)
        img = (img * 255).astype(np.uint8)
    if img.ndim == 2:
        img = np.stack([img]*3, axis=-1)
    if img.shape[2] == 4:
        img = img[..., :3]
    img_h, img_w = img.shape[:2]
    # arrays
    triangles = np.asarray(mesh.triangles, dtype=np.int32)
    tri_uvs = np.asarray(mesh.triangle_uvs, dtype=np.float32).reshape((-1, 3, 2))  # (n_tri,3,2)
    n_vertices = len(mesh.vertices)
    sums = np.zeros((n_vertices, 3), dtype=np.float64)
    counts = np.zeros((n_vertices,), dtype=np.int32)

    n_tri = triangles.shape[0]
    for start in range(0, n_tri, batch_faces):
        end = min(n_tri, start + batch_faces)
        tris = triangles[start:end]
        uvs = tri_uvs[start:end].reshape((-1, 2))  # flattened per-vertex
        # sample colors for each triangle vertex
        colors = sample_texture_bilinear(img, uvs)  # shape (batch_faces*3, 3)
        colors = colors.reshape((-1, 3))  # (n_batch*3,3)
        # accumulate per vertex
        for i in range(end - start):
            tri = tris[i]
            c0 = colors[3*i + 0]
            c1 = colors[3*i + 1]
            c2 = colors[3*i + 2]
            sums[tri[0]] += c0
            sums[tri[1]] += c1
            sums[tri[2]] += c2
            counts[tri[0]] += 1
            counts[tri[1]] += 1
            counts[tri[2]] += 1
        gc.collect()
        print(f"Processed triangles {start}..{end} / {n_tri}")

    # avoid division by zero
    mask = counts > 0
    vertex_colors = np.zeros((n_vertices, 3), dtype=np.float32)
    vertex_colors[mask] = (sums[mask] / counts[mask][:, None]).astype(np.float32)
    # for vertices with zero counts, set to gray
    vertex_colors[~mask] = 0.5
    return vertex_colors

# Try baking if possible
try:
    if tex_path and len(mesh.triangle_uvs) > 0:
        vcols = bake_vertex_colors_from_texture(mesh, tex_path)
        mesh.vertex_colors = o3d.utility.Vector3dVector(vcols)
        o3d.io.write_triangle_mesh(str(out_ply), mesh, write_vertex_colors=True)
        print(f"Saved colored mesh to: {out_ply}")
    else:
        raise RuntimeError("No texture or UVs available for baking.")
except Exception as e:
    print(f"Baking failed: {e}")
    print("Falling back to sampling colored point cloud from textured mesh (slower but robust).")
    try:
        # fallback: sample points with colors (requires texture accessible by Open3D)
        pcd = mesh.sample_points_poisson_disk(number_of_points=300000)
        if pcd.has_colors():
            o3d.io.write_point_cloud(str(out_ply), pcd, write_ascii=False)
            print(f"Saved colored point cloud to: {out_ply}")
        else:
            # final fallback: save geometry only
            o3d.io.write_triangle_mesh(str(out_ply), mesh, write_vertex_colors=False)
            print(f"Saved mesh without colors to: {out_ply} (no colors available)")
    except Exception as e2:
        print(f"Fallback also failed: {e2}")
        # still attempt to write mesh
        o3d.io.write_triangle_mesh(str(out_ply), mesh, write_vertex_colors=False)
        print(f"Wrote mesh without colors to: {out_ply}")

# Visualize the resulting PLY (or mesh)
if out_ply.exists():
    geom = o3d.io.read_triangle_mesh(str(out_ply))
    print("Result mesh info:", len(geom.vertices), "vertices,", len(geom.triangles), "faces")
    if geom.has_vertex_colors():
        print("Result has vertex colors.")
    elif geom.has_textures():
        print("Result has textures.")
    else:
        print("Result has no vertex colors or textures.")
    o3d.visualization.draw_geometries([geom], window_name='TexturedMesh (baked)')

Using texture: C:\Users\gnoceras\Documents\GustavoPersonal\ReconstructionStudies\results\meshroom_rec\texture_1001.png
Processed triangles 0..42606 / 42606
[Open3D WARNING] This file format currently does not support writing textures and uv coordinates. Consider using .obj
Saved colored mesh to: ..\results\meshroom_rec\reconstruction.ply
Result mesh info: 127806 vertices, 42606 faces
Result has vertex colors.


## Convert to Point Cloud

In [ ]:
# Convert Mesh to Point Cloud

mesh = o3d.io.read_triangle_mesh(str(out_ply))
verts = np.asarray(mesh.vertices)
tris  = np.asarray(mesh.triangles)

# optional per-vertex colors (if present)
has_colors = mesh.has_vertex_colors()
vcols = np.asarray(mesh.vertex_colors) if has_colors else None

# face areas
f0 = verts[tris[:,0]]
f1 = verts[tris[:,1]]
f2 = verts[tris[:,2]]

areas = 0.5 * np.linalg.norm(np.cross(f1 - f0, f2 - f0), axis=1)
probs = areas / areas.sum()

def sample_points_on_mesh(n_points):
    face_idx = np.random.choice(len(tris), size=n_points, p=probs)
    r1 = np.sqrt(np.random.rand(n_points))
    r2 = np.random.rand(n_points)
    a = 1 - r1
    b = r1 * (1 - r2)
    c = r1 * r2
    p = (a[:,None]*verts[tris[face_idx,0]] +
         b[:,None]*verts[tris[face_idx,1]] +
         c[:,None]*verts[tris[face_idx,2]])
    if has_colors:
        color = (a[:,None]*vcols[tris[face_idx,0]] +
                 b[:,None]*vcols[tris[face_idx,1]] +
                 c[:,None]*vcols[tris[face_idx,2]])
    else:
        color = None
    return p, color

pts, cols = sample_points_on_mesh(300_000)
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(pts)
if cols is not None:
    pcd.colors = o3d.utility.Vector3dVector(cols)
o3d.io.write_point_cloud("../results/meshroom_rec/pointcloud.ply", pcd)
o3d.visualization.draw_geometries([pcd])

Mesh has vertex colors.
